# Machine Learning based Retrieval of Ground-Level Black Carbon (BC) Concentrations
### Satellite Observations Validated with Reanalysis and Aethalometer Datasets

**Pilot Study Area:** Lahore Metropolitan Region ($31.0^\circ\text{N} - 32.0^\circ\text{N},\ 73.8^\circ\text{E} - 74.8^\circ\text{E}$)  
**Monitoring Site:** Lahore Ground Station ($31.4792^\circ\text{N},\ 74.2649^\circ\text{E}$)  
**Instrumentation:** Magee Scientific AE33 Aethalometer, Sentinel-5P TROPOMI UVAI, MODIS MAIAC 1 km AOD, NASA MERRA-2 Reanalysis.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Add project root
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from configs.config import PROCESSED_DATA_DIR, OUTPUTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR
print(f"Project root loaded: {ROOT_DIR}")

## 1. Load Processed Collocated Ground & Satellite Dataset
We load the collocated dataset matching ground AE33 measurements, Sentinel-5P TROPOMI UVAI, and NASA MERRA-2 aerosol/meteorology fields.

In [ ]:
data_path = PROCESSED_DATA_DIR / "collocated_bc_hourly.csv"
df = pd.read_csv(data_path)
df["datetime_utc"] = pd.to_datetime(df["datetime_utc"])
print(f"Collocated dataset loaded: {len(df)} samples across {df.shape[1]} features.")
df[["datetime_utc", "BC_target_ug_m3", "merra2_bc_smass_ug_m3", "merra2_wind_speed", "merra2_pblh"]].head()

## 2. Quantitative Model Performance Benchmarks
Display quantitative held-out test benchmarks across candidate retrieval architectures.

In [ ]:
benchmarks_path = TABLES_DIR / "heldout_test_benchmarks.csv"
df_bench = pd.read_csv(benchmarks_path, index_col=0)
df_bench

In [ ]:
# Visualizing R2 and RMSE comparison across models
fig, ax1 = plt.subplots(figsize=(9, 4.5), dpi=150)
x = np.arange(len(df_bench))
width = 0.35

rects1 = ax1.bar(x - width/2, df_bench["R2"], width, label="R² Score", color="#2b5c8f")
ax1.set_ylabel("R² Score", fontsize=11, fontweight="bold", color="#2b5c8f")
ax1.set_ylim(0.6, 1.0)

ax2 = ax1.twinx()
rects2 = ax2.bar(x + width/2, df_bench["RMSE"], width, label="RMSE (µg/m³)", color="#e76f51")
ax2.set_ylabel("RMSE (µg/m³)", fontsize=11, fontweight="bold", color="#e76f51")

ax1.set_xticks(x)
ax1.set_xticklabels(df_bench.index, rotation=20, ha="right", fontsize=10)
ax1.set_title("Independent Held-Out Validation Performance Across Models", fontsize=12, fontweight="bold")
fig.tight_layout()
plt.show()

## 3. High-Resolution Pilot Area Hotspot Map
We visualize the high-resolution gridded surface BC retrieval generated by the best-performing machine learning model.

In [ ]:
from IPython.display import Image
hotspot_img = FIGURES_DIR / "pilot_area_bc_hotspot_map.png"
Image(filename=str(hotspot_img))

## 4. Key Physical Predictors (Feature Importance)
Ranking the top atmospheric and satellite predictors driving surface Black Carbon concentration.

In [ ]:
feat_img = FIGURES_DIR / "feature_importance_best_model.png"
Image(filename=str(feat_img))